# 14. Native-600px loss sweep -- every U-Net-family arm at full resolution

Extends 05 and 13's loss-fn sweep (MAE/wavelet/starlet/gradient, 2026-09-15) across the
full pixel resolution instead of the 256px round trip every other checkpoint in this
project trains at. **This has never been done before at this scale.** `winner_native600`
(05, single arm, original loss only) has never once completed a full run (RULES.md #1) --
this notebook runs 40 arms at the same resolution, three of them (the `kin_gamma0`
arms) at 31 input channels, ~31x the tensor size of the case that already never finished.
Going in with that understood, not discovered after the fact.

**Scope**, all fine-tuned from the best confirmed checkpoint of each family plus a fresh-
init counterpart, same ablation split as 05/13:

| section | source | channels | arms |
|---|---|---|---|
| 2 | `winner_aug_seed43`, `winner_p10_seed44`, `winner_beam_seed42` | 1 | 4 losses x 2 x 3 = 24 |
| 3 | `kin_gamma0` | 31 | 4 losses x 2 = 8 |
| 4 | `sg_k3_fresh` | 7 | 4 losses x 2 = 8 |

**Not in this notebook**: `ddpm_seed42`/`ddrm_prior` at native resolution. Their
architecture (`ch_mult=[1,2,2,2,4]`, `attn_resolutions=[16]`) is tuned for 256px's specific
downsampling schedule -- at 600px input the same depth lands on a different feature-map
size, so this needs an architecture redesign, not a `target_size` change. Not attempted
here; a wrong version would be silently-broken, not just slow.

`MAX_NEW_ARMS_PER_SESSION = 1` -- the most conservative setting in the project, given zero
of this notebook's arms have ever completed once. Expect many sessions; expect some arms
(`kin_gamma0` especially) to simply not fit in T4 memory regardless of batch size, which is
a legitimate result to report, not a bug to chase (RULES.md's own standard: a negative
result, named honestly, is a finding).


## 0. Bootstrap

In [ ]:
import os, sys, subprocess, glob, re

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'native600-loss-sweep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    run_re = re.compile(r'run_\d+_\d+_rt_\d+', re.I)
    roots = {}
    for p in glob.glob('/kaggle/input/**/run_*', recursive=True):
        if os.path.isdir(p) and run_re.search(os.path.basename(p)):
            roots[os.path.dirname(p)] = roots.get(os.path.dirname(p), 0) + 1
    if not roots:
        raise FileNotFoundError('No run_<id>_<step>_rt_<pp> folders under /kaggle/input. '
                                'Attach the line-emission Dataset.')
    DATA_DIR = max(roots, key=roots.get)

    sg_hits = [p for p in glob.glob('/kaggle/input/**/run_9*_rt_*', recursive=True)
              if os.path.isdir(p)]
    SG_DATA_DIR = os.path.dirname(sg_hits[0]) if sg_hits else None

    # best_models sources, uploaded under a neutral extension (RULES.md #3). Same 4
    # sources 05/13 already use: winner_aug_seed43, winner_p10_seed44, winner_beam_seed42,
    # kin_gamma0, sg_k3_fresh -- reuse whichever dataset(s) already have them attached.
    def locate_ckpt(stem):
        hits = [h for ext in ('.pth', '.ckpt', '.pth.tar')
               for h in glob.glob(f'/kaggle/input/**/{stem}{ext}', recursive=True)
               if os.path.isfile(h)]
        return hits[0] if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../Line Emission Data'
    SG_DATA_DIR = '../self-gravitating cube and dirty cube/sg_synth'
    def locate_ckpt(stem):
        hits = glob.glob(f'../models/best_models/{stem}.pth')
        return hits[0] if hits else None

print('DATA_DIR   :', DATA_DIR)
print('SG_DATA_DIR:', SG_DATA_DIR or 'NOT FOUND -- section 4 will be skipped')


## 0b. Pull latest `src` (re-run anytime -- no kernel restart needed)

In [ ]:
if ON_KAGGLE:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'], capture_output=True, text=True).stdout)
    import importlib, src; importlib.reload(src)


## 1. Imports, shared config

In [ ]:
import time, csv, shutil, math
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes, list_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet, LOSS_REGISTRY

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

NATIVE_SIZE = 600
LOSSES = ['mae', 'wavelet', 'starlet', 'gradient']

# Nothing in this notebook has ever completed once. 1 arm/session is the most
# conservative setting anywhere in this project on purpose.
MAX_NEW_ARMS_PER_SESSION = 1
_new_arms_trained = 0
# See sweep.py / 05 / 13 for why -- fine-tune arms at full lr spike and knock out the
# pretrained weights within a few epochs.
FINETUNE_LR_SCALE = 0.1


def _cap_reached(name):
    if MAX_NEW_ARMS_PER_SESSION and _new_arms_trained >= MAX_NEW_ARMS_PER_SESSION:
        print(f'--- {name}: DEFERRED, session cap of {MAX_NEW_ARMS_PER_SESSION} new arms '
              f'reached -- resumes next session ---', flush=True)
        return True
    return False


OUT_DIR = '../results'
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def persist_ckpt(path, note='', csv_path=None):
    """Copy a finished checkpoint (and its CSV) to /kaggle/working the moment it exists
    (RULES.md #1) -- CKPT_DIR/OUT_DIR are wiped by the next session's bootstrap and are
    NOT part of the notebook Output. At this notebook's per-arm cost (hours, not minutes),
    losing one to a late persist is the single most expensive mistake available here."""
    if not (ON_KAGGLE and path and os.path.exists(path)):
        return None
    base = os.path.basename(path)
    dst = os.path.join('/kaggle/working', base[:-4] if base.endswith('.pth.tar') else base)
    shutil.copy2(path, dst)
    if csv_path and os.path.exists(csv_path):
        shutil.copy2(csv_path, os.path.join('/kaggle/working', os.path.basename(csv_path)))
    print(f'    [persisted] {os.path.basename(dst)} ({os.path.getsize(dst)/1e6:.0f} MB)'
         + (f' -- {note}' if note else ''), flush=True)
    return dst


def _import_prior_nb14():
    """Restore checkpoints + CSVs from an earlier session's Output. Same reasoning as
    05's `_import_prior_nb05` / 13's `_import_prior_nb13` -- without this every session
    retrains from scratch, and at this notebook's per-arm cost that is not affordable."""
    if not ON_KAGGLE:
        return
    n_ck = 0
    for ext in ('.pth', '.ckpt', '.pth.tar'):
        for src in sorted(glob.glob(f'/kaggle/input/**/nb14_*{ext}', recursive=True)):
            if not os.path.isfile(src):
                continue
            dst = os.path.join(CKPT_DIR, os.path.basename(src)[:-len(ext)] + '.pth')
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                n_ck += 1
    n_csv = 0
    for csv_name in ('nb14_loss_sweep.csv', 'nb14_kin_loss_sweep.csv', 'nb14_sg_loss_sweep.csv'):
        dst = os.path.join(OUT_DIR, csv_name)
        if os.path.exists(dst):
            continue
        hits = glob.glob(f'/kaggle/input/**/{csv_name}', recursive=True)
        if hits:
            shutil.copy2(hits[0], dst)
            n_csv += 1
    if n_ck or n_csv:
        print(f'[nb14 prior] restored {n_ck} checkpoint(s) and {n_csv} CSV(s) from a prior Output')


def _done_rows(path, fields, ext='.pth'):
    """Rows whose CHECKPOINT is also present -- a CSV row alone is not proof an arm is
    done (RULES.md #12; caught in 13, same bug, same fix)."""
    out = {}
    if os.path.exists(path):
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                name = r['config']
                if not os.path.exists(os.path.join(CKPT_DIR, f'nb14_{name}{ext}')):
                    print(f'[resume] {name}: CSV row found but no checkpoint -- will retrain')
                    continue
                out[name] = {k: r.get(k, '') for k in fields}
    return out


_import_prior_nb14()
print(f'device: {device} | GPUs: {N_GPU} | native size: {NATIVE_SIZE} | losses: {LOSSES}')


## 2. Line-emission loss sweep at 600px -- `winner_aug_seed43` / `winner_p10_seed44` /
`winner_beam_seed42`, 4 losses x fine-tune/fresh x 3 sources = 24 arms

Same WINNER hyperparameters as every other arm in this family. `batch_size=4` -- the
convention `winner_native600` already established (05: 5.5x a 256px epoch's pixel volume).


In [ ]:
WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=8.196504330730313e-4, alpha=0.8877681051398497,
              sched_patience=8, batch_size=4)
N_SAMPLES = 100   # fewer than 05's 150 -- 600px items are already the memory pressure,
                  # no need to also inflate how many are indexed per epoch
NW = 2 if torch.cuda.is_available() else 0

train_cubes, val_cubes, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                        val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=N_SAMPLES, target_size=NATIVE_SIZE, seed=SEED,
          subtract_continuum=True, continuum_n=5, verbose=False)
train_ds = FITSChannelDataset(train_cubes, **_kw)
val_ds   = FITSChannelDataset(val_cubes, **_kw)
train_ds_beam = FITSChannelDataset(train_cubes, return_beam=True, **_kw)
val_ds_beam   = FITSChannelDataset(val_cubes, return_beam=True, **_kw)
print(f'600px line-emission: train {len(train_ds)} | val {len(val_ds)}')


In [ ]:
SOURCES = {
    'aug':  ('winner_aug_seed43',  'full'),
    'p10':  ('winner_p10_seed44',  'full'),
    'beam': ('winner_beam_seed42', 'beam'),
}
SRC_CKPT = {tag: locate_ckpt(stem) for tag, (stem, _) in SOURCES.items()}
for tag, path in SRC_CKPT.items():
    print(f'{tag} source ({SOURCES[tag][0]}):', path or 'NOT FOUND -- fine-tune arms for this source fall back to fresh init')

FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
         'epochs_run', 'wall_time_s']
LOSS_CSV = os.path.join(OUT_DIR, 'nb14_loss_sweep.csv')
loss_done = _done_rows(LOSS_CSV, FIELDS)
if loss_done:
    print(f'[resume] {len(loss_done)} arm(s) already scored: {sorted(loss_done)}')

loss_rows = list(loss_done.values())
for src_tag, (src_stem, view) in SOURCES.items():
    use_beam = view == 'beam'
    tr, va = (train_ds_beam, val_ds_beam) if use_beam else (train_ds, val_ds)
    for loss_name in LOSSES:
        for source in ('finetune', 'fresh'):
            tag = 'ft' if source == 'finetune' else source
            name = f'winner_{loss_name}_{src_tag}_{tag}_600'
            if name in loss_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
            init_state = None
            if source == 'finetune':
                if SRC_CKPT[src_tag] is None:
                    print(f'--- {name}: SKIPPED, no source checkpoint for {src_tag} ---')
                    continue
                init_state = torch.load(SRC_CKPT[src_tag], map_location=device,
                                        weights_only=False)['model_state_dict']
                _min_ep = 15   # 600px epochs are ~6x a 256px epoch's wall time --
                               # 30/50 (05's budget) would be unaffordable here
                _lr = WINNER['lr'] * FINETUNE_LR_SCALE
            else:
                _min_ep = 25
                _lr = WINNER['lr']
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            res = train_unet(tr, va, device, **{**WINNER, 'lr': _lr, 'loss_name': loss_name,
                                                'use_beam': use_beam},
                             min_epochs=_min_ep, max_epochs=35, patience=6,
                             num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                             init_state_dict=init_state)
            row = {'config': name, 'source': source,
                   **{k: res[k] for k in FIELDS if k in res}}
            loss_rows.append(row)
            new = not os.path.exists(LOSS_CSV)
            with open(LOSS_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in FIELDS})
            persist_ckpt(ckpt, name, csv_path=LOSS_CSV)
            _new_arms_trained += 1
            print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
            res.pop('model', None)
            if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n600px line-emission sweep: {len(loss_rows)} row(s)')


## 3. `kin_gamma0` loss sweep at 600px -- 31 input channels, 4 losses x ft/fresh = 8 arms

**The highest-risk section in this notebook.** 31 channels x 600 x 600 is ~31x the tensor
size of `winner_native600` (1 channel), which has never completed once at batch_size=4.
`batch_size=1` here is not a tuning choice, it's the floor -- if this still does not fit in
a T4's 16 GB, that is a real answer (kin_gamma0 does not scale to native resolution on this
hardware), not a bug to keep chasing. `N_SAMPLES` and `min_epochs` both cut well below
section 2's budget for the same reason: this section is about finding out whether it runs
at all before spending real epochs on it.


In [ ]:
K_KIN = 15
N_CH_KIN = 2 * K_KIN + 1
N_SAMPLES_KIN = 40

KIN_WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                  lr=8.196504330730313e-4, alpha=0.8877681051398497,
                  sched_patience=8, batch_size=1,
                  n_neighbors=K_KIN, out_channels=N_CH_KIN, kinematic_gamma=0.0)

train_cubes_kin, val_cubes_kin, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=N_SAMPLES_KIN, target_size=NATIVE_SIZE, seed=SEED,
          subtract_continuum=True, continuum_n=5,
          n_neighbors=K_KIN, stack_target=True, verbose=False)
train_ds_kin = FITSChannelDataset(train_cubes_kin, **_kw)
val_ds_kin   = FITSChannelDataset(val_cubes_kin, **_kw)
d, c = train_ds_kin[0]
assert d.shape == c.shape == (N_CH_KIN, NATIVE_SIZE, NATIVE_SIZE)
print(f'kin 600px: train {len(train_ds_kin)} | val {len(val_ds_kin)} | '
     f'{N_CH_KIN}-channel stacks | one item = {d.numel() * 4 / 1e6:.0f} MB (dirty alone)')

from astropy.io import fits
_h = fits.getheader(train_cubes_kin[0]['clean'])
VELAX_KIN = (np.arange(N_CH_KIN) - K_KIN) * float(_h['CDELT3'])


In [ ]:
KIN_CKPT_SRC = locate_ckpt('kin_gamma0')
print('kin_gamma0 source:', KIN_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

KIN_FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
             'epochs_run', 'wall_time_s']
KIN_CSV = os.path.join(OUT_DIR, 'nb14_kin_loss_sweep.csv')
kin_done = _done_rows(KIN_CSV, KIN_FIELDS)
if kin_done:
    print(f'[resume] {len(kin_done)} kin arm(s) already scored: {sorted(kin_done)}')

kin_rows = list(kin_done.values())
for loss_name in LOSSES:
    for source in ('finetune', 'fresh'):
        tag = 'ft' if source == 'finetune' else source
        name = f'kin_gamma0_{loss_name}_{tag}_600'
        if name in kin_done:
            print(f'--- {name}: SKIPPED, already done ---')
            continue
        if _cap_reached(name):
            continue
        ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
        init_state = None
        if source == 'finetune':
            if KIN_CKPT_SRC is None:
                print(f'--- {name}: SKIPPED, no source checkpoint ---')
                continue
            init_state = torch.load(KIN_CKPT_SRC, map_location=device,
                                    weights_only=False)['model_state_dict']
            _min_ep, _lr = 8, KIN_WINNER['lr'] * FINETUNE_LR_SCALE
        else:
            _min_ep, _lr = 15, KIN_WINNER['lr']
        print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
        try:
            res = train_unet(train_ds_kin, val_ds_kin, device,
                             **{**KIN_WINNER, 'lr': _lr, 'loss_name': loss_name},
                             velax_kms=VELAX_KIN, min_epochs=_min_ep, max_epochs=25, patience=5,
                             num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                             init_state_dict=init_state)
        except RuntimeError as e:
            # Catches RuntimeError, not just torch.cuda.OutOfMemoryError (a subclass only
            # in newer torch -- older versions raise a plain RuntimeError whose message
            # says 'out of memory'). A real answer, not a bug: 31ch x 600x600 may simply
            # not fit on a T4 even at batch_size=1. Record it and move on rather than let
            # one arm's OOM kill the whole session.
            if 'out of memory' not in str(e).lower():
                raise
            print(f'  [OOM] {name} does not fit in GPU memory at batch_size=1: {e}')
            torch.cuda.empty_cache()
            continue
        row = {'config': name, 'source': source,
               **{k: res[k] for k in KIN_FIELDS if k in res}}
        kin_rows.append(row)
        new = not os.path.exists(KIN_CSV)
        with open(KIN_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=KIN_FIELDS)
            if new: w.writeheader()
            w.writerow({k: row.get(k, '') for k in KIN_FIELDS})
        persist_ckpt(ckpt, name, csv_path=KIN_CSV)
        _new_arms_trained += 1
        print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
        res.pop('model', None)
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nkin_gamma0 600px sweep: {len(kin_rows)} row(s)')


## 4. `sg_k3_fresh` loss sweep at 600px -- 7 input channels, 4 losses x ft/fresh = 8 arms

SG cubes are also native 601x600x600 (`results/self-gravitating/README.md`), so this is the
same mechanical change as section 2 -- 7 channels is much closer to section 2's 1-channel
case than to section 3's 31-channel one. `batch_size=2`, between section 2's 4 and section
3's 1.


In [ ]:
if SG_DATA_DIR is None:
    print('SG_DATA_DIR not found -- section 4 skipped.')
else:
    K_SG = 3
    N_SAMPLES_SG = 60

    SG_WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                     lr=8.196504330730313e-4, alpha=0.8877681051398497,
                     sched_patience=8, batch_size=2,
                     n_neighbors=K_SG, out_channels=1)

    TRAIN_RUNS_SG = ['run_9015_00370_rt_00', 'run_9019_00019_rt_00', 'run_9032_00020_rt_00']
    VAL_RUN_SG = 'run_9025_00370_rt_00'

    all_cubes_sg = {c['folder']: c for c in list_cubes(SG_DATA_DIR)}
    train_cubes_sg = [all_cubes_sg[r] for r in TRAIN_RUNS_SG]
    val_cubes_sg = [all_cubes_sg[VAL_RUN_SG]]

    _kw = dict(n_samples=N_SAMPLES_SG, target_size=NATIVE_SIZE, seed=SEED,
              subtract_continuum=False, n_neighbors=K_SG, stack_target=False, verbose=False)
    train_ds_sg = FITSChannelDataset(train_cubes_sg, **_kw)
    val_ds_sg   = FITSChannelDataset(val_cubes_sg, **_kw)
    d, c = train_ds_sg[0]
    assert d.shape[0] == 2 * K_SG + 1 and c.shape[0] == 1
    print(f'sg 600px: train {len(train_ds_sg)} | val {len(val_ds_sg)} | '
         f'{2*K_SG+1}-channel input, 1-ch target')


In [ ]:
if SG_DATA_DIR is None:
    print('section 4 skipped')
else:
    SG_CKPT_SRC = locate_ckpt('sg_k3_fresh')
    print('sg_k3_fresh source:', SG_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

    SG_FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
                'epochs_run', 'wall_time_s']
    SG_CSV = os.path.join(OUT_DIR, 'nb14_sg_loss_sweep.csv')
    sg_done = _done_rows(SG_CSV, SG_FIELDS)
    if sg_done:
        print(f'[resume] {len(sg_done)} sg arm(s) already scored: {sorted(sg_done)}')

    sg_rows = list(sg_done.values())
    for loss_name in LOSSES:
        for source in ('finetune', 'fresh'):
            tag = 'ft' if source == 'finetune' else source
            name = f'sg_k3_{loss_name}_{tag}_600'
            if name in sg_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
            init_state = None
            if source == 'finetune':
                if SG_CKPT_SRC is None:
                    print(f'--- {name}: SKIPPED, no source checkpoint ---')
                    continue
                init_state = torch.load(SG_CKPT_SRC, map_location=device,
                                        weights_only=False)['model_state_dict']
                _min_ep, _lr = 10, SG_WINNER['lr'] * FINETUNE_LR_SCALE
            else:
                _min_ep, _lr = 20, SG_WINNER['lr']
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            try:
                res = train_unet(train_ds_sg, val_ds_sg, device,
                                 **{**SG_WINNER, 'lr': _lr, 'loss_name': loss_name},
                                 min_epochs=_min_ep, max_epochs=35, patience=6,
                                 num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                                 init_state_dict=init_state)
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower():
                    raise
                print(f'  [OOM] {name} does not fit in GPU memory at batch_size=2: {e}')
                torch.cuda.empty_cache()
                continue
            row = {'config': name, 'source': source,
                   **{k: res[k] for k in SG_FIELDS if k in res}}
            sg_rows.append(row)
            new = not os.path.exists(SG_CSV)
            with open(SG_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=SG_FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in SG_FIELDS})
            persist_ckpt(ckpt, name, csv_path=SG_CSV)
            _new_arms_trained += 1
            print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
            res.pop('model', None)
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nsg_k3_fresh 600px sweep: {len(sg_rows)} row(s)')


## 5. Collect outputs

In [ ]:
from src.evaluation.collect_outputs import collect_outputs

_run_dir = collect_outputs(
    '14-native600-loss-sweep',
    [
        'nb14_loss_sweep.csv',
        'nb14_kin_loss_sweep.csv',
        'nb14_sg_loss_sweep.csv',
    ],
    extra={'losses': LOSSES, 'native_size': NATIVE_SIZE,
          'sg_ckpt_found': SG_DATA_DIR is not None and 'SG_CKPT_SRC' in dir() and SG_CKPT_SRC is not None},
)
print('\nAnything listed as NOT FOUND above did not get written this run.')
